# Silver Layer, Orders Data Processing
**Source:** Bronze Delta Table `bronze.bronze_transactions`  
**Target:** Silver Delta Table `/delta/silver/orders` (Hive table `silver.silver_orders`)  
**Pattern:** Incremental Load with Merge (UPSERT) based on `transaction_id`  
**Transformations:**
- Normalize quantity and total_amount (negative values set to 0)
- Cast `transaction_date` to DATE type
- Derive `order_status` (Cancelled if quantity or total_amount = 0, else Completed)
- Filter out records with null `transaction_date`, `customer_id`, or `product_id`
- Add `last_updated` timestamp
  
**Layer:** Silver

In [2]:
import os, sys, logging
from pyspark.sql import SparkSession

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "bronze_db"      : "bronze",
    "bronze_table"   : "bronze_transactions",
    "silver_db"      : "silver",
    "silver_table"   : "silver_orders",
    "silver_path"    : "hdfs://master:8020/delta/silver/orders",
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Silver_Orders_Load")
        .master("spark://master:7077")
        .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.2.1")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .enableHiveSupport()
        .getOrCreate()
    )

def create_silver_table_if_not_exists(spark):
    """Create the silver orders table if it doesn't exist."""
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CONFIG['silver_db']}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CONFIG['silver_db']}.{CONFIG['silver_table']} (
            transaction_id STRING,
            customer_id STRING,
            product_id STRING,
            quantity INT,
            total_amount DOUBLE,
            transaction_date DATE,
            payment_method STRING,
            store_type STRING,
            order_status STRING,
            last_updated TIMESTAMP
        )
        USING DELTA
        LOCATION '{CONFIG['silver_path']}'
    """)
    logger.info(f"Ensured table {CONFIG['silver_db']}.{CONFIG['silver_table']} exists.")

def get_last_processed_timestamp(spark):
    """Return the maximum last_updated timestamp from silver table, or a default."""
    try:
        df = spark.sql(f"SELECT MAX(last_updated) AS last_processed FROM {CONFIG['silver_db']}.{CONFIG['silver_table']}")
        row = df.collect()[0]
        ts = row['last_processed']
        if ts is None:
            return "1900-01-01T00:00:00.000+00:00"
        else:
            return ts.isoformat() if hasattr(ts, 'isoformat') else str(ts)
    except Exception as e:
        logger.warning(f"Could not read last_processed (table may be empty): {e}")
        return "1900-01-01T00:00:00.000+00:00"

def run():
    logger.info("Silver Orders pipeline started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    create_silver_table_if_not_exists(spark)

    last_processed = get_last_processed_timestamp(spark)
    logger.info(f"Last processed timestamp: {last_processed}")

    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW bronze_incremental AS
        SELECT *
        FROM {CONFIG['bronze_db']}.{CONFIG['bronze_table']}
        WHERE ingestion_timestamp > TIMESTAMP '{last_processed}'
    """)

    # Check if there is any new data
    count_new = spark.table("bronze_incremental").count()
    if count_new == 0:
        logger.info("No new data to process. Exiting.")
        return
    logger.info(f"Found {count_new} new rows in bronze.")

    spark.sql("""
        CREATE OR REPLACE TEMP VIEW silver_incremental AS
        SELECT
            transaction_id,
            customer_id,
            product_id,
            CASE WHEN quantity < 0 THEN 0 ELSE quantity END AS quantity,
            CASE WHEN total_amount < 0 THEN 0 ELSE total_amount END AS total_amount,
            CAST(transaction_date AS DATE) AS transaction_date,
            payment_method,
            store_type,
            CASE
                WHEN quantity = 0 OR total_amount = 0 THEN 'Cancelled'
                ELSE 'Completed'
            END AS order_status,
            CURRENT_TIMESTAMP() AS last_updated
        FROM bronze_incremental
        WHERE transaction_date IS NOT NULL
          AND customer_id IS NOT NULL
          AND product_id IS NOT NULL
    """)

    logger.info("Sample of transformed silver data:")
    spark.table("silver_incremental").show(5, truncate=False)

    # Merge into silver table
    merge_result = spark.sql(f"""
        MERGE INTO {CONFIG['silver_db']}.{CONFIG['silver_table']} AS target
        USING silver_incremental AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    logger.info(f"Merge completed: {merge_result.first()['num_affected_rows']} rows affected.")
    final_count = spark.table(f"{CONFIG['silver_db']}.{CONFIG['silver_table']}").count()
    logger.info(f"SUCCESS: Silver orders table now has {final_count} rows.")

if __name__ == "__main__":
    run()

2026-03-19 13:18:01,566 [INFO] Silver Orders pipeline started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-370dab4c-afa5-4475-943f-8aad73220802;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (4705ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.1/delta-storage-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.1!delta-storage.jar (290ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (407ms)
:: resolution report :: resolve 2635ms

+--------------+-----------+----------+--------+------------+----------------+--------------+--------------+------------+-----------------------+
|transaction_id|customer_id|product_id|quantity|total_amount|transaction_date|payment_method|store_type    |order_status|last_updated           |
+--------------+-----------+----------+--------+------------+----------------+--------------+--------------+------------+-----------------------+
|TRX000001     |802        |425       |1       |363.4       |2020-07-27      |Debit Card    |Physical Store|Completed   |2026-03-19 13:19:18.463|
|TRX000002     |858        |280       |6       |758.18      |2022-08-10      |Credit Card   |Physical Store|Completed   |2026-03-19 13:19:18.463|
|TRX000003     |658        |694       |9       |748.66      |2020-05-22      |Bank Transfer |Online        |Completed   |2026-03-19 13:19:18.463|
|TRX000005     |368        |104       |10      |137.28      |2022-06-24      |PayPal        |Physical Store|Completed   |202

2026-03-19 13:19:29,414 [INFO] Merge completed: 9714 rows affected.             
2026-03-19 13:19:37,698 [INFO] SUCCESS: Silver orders table now has 9714 rows.  
